In [12]:
# Step 1:
import pandas as pd
import re

# Step 2: Prepare Search Content & Clean All Columns
# Step 1: Load Dataset directly from GitHub Raw Link
github_raw_url = 'https://raw.githubusercontent.com/imhasannnn/Data-Mining-Lab/main/imdb_clean.csv'

try:
    df = pd.read_csv(github_raw_url)
    print("Dataset loaded successfully from GitHub Remote Repository!")
except Exception as e:
    # Fallback to local files if internet is unavailable or run in local directory
    try:
        df = pd.read_csv('/content/imdb_clean.csv')
        print("Dataset loaded successfully from Colab directory.")
    except FileNotFoundError:
        df = pd.read_csv('imdb_clean.csv')
        print("Dataset loaded successfully from local directory.")

# Handle missing data cleanly
df['title'] = df['title'].fillna('Unknown')
df['director'] = df['director'].fillna('Unknown')
df['release_year'] = df['release_year'].fillna('N/A')
df['runtime'] = df['runtime'].fillna('N/A')
df['genre'] = df['genre'].fillna('Unknown')
df['rating'] = df['rating'].fillna('N/A')
df['metascore'] = df['metascore'].fillna('N/A')
df['gross(M)'] = df['gross(M)'].fillna('N/A')

# Combine all searchable fields into a single column
df['search_content'] = (
    df['title'] + " " +
    df['director'] + " " +
    df['genre'] + " " +
    df['release_year'].astype(str) + " " +
    df['runtime'].astype(str) + " " +
    df['rating'].astype(str)
)

# Step 3: Preprocessing Function
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s.]', '', text) # Keeping '.' to support ratings like 9.3
    return text.split()

# Step 4: Build Inverted Index
inverted_index = {}
for movie_id, row in df.iterrows():
    tokens = preprocess_text(row['search_content'])
    for token in set(tokens):
        if token not in inverted_index:
            inverted_index[token] = []
        inverted_index[token].append(movie_id)

# Step 5: Boolean Search Function
def search_movies(query, mode='AND'):
    query_tokens = preprocess_text(query)
    if not query_tokens:
        return []

    first_token = query_tokens[0]
    result_set = set(inverted_index.get(first_token, []))

    for token in query_tokens[1:]:
        current_movie_ids = set(inverted_index.get(token, []))
        if mode == 'AND':
            result_set = result_set.intersection(current_movie_ids)
        else:
            result_set = result_set.union(current_movie_ids)

    return list(result_set)

# Step 6: Ranking, Display, and Smart Insights (The Unique Feature)
def ranked_search(query, mode='AND', is_multi_word=False):
    matched_movie_ids = search_movies(query, mode=mode)
    query_tokens = preprocess_text(query)

    ranked_results = []
    for movie_id in matched_movie_ids:
        movie_text = df.iloc[movie_id]['search_content'].lower()
        score = sum(movie_text.count(token) for token in query_tokens)
        ranked_results.append((movie_id, score))

    ranked_results.sort(key=lambda x: x[1], reverse=True)

    if is_multi_word:
        print(f"\nResults for: '{query}' (Mode: {mode})")
    else:
        print(f"\nResults for: '{query}'")

    print("-" * 50)

    if not ranked_results:
        print("No results found.")
        return

    # Lists to store numeric data for generating smart insights
    valid_earnings = []
    valid_ratings = []
    highest_earning_movie = "N/A"
    max_earnings = -1.0

    # Display Top 5 Results
    for rank, (movie_id, score) in enumerate(ranked_results[:5], 1):
        row = df.iloc[movie_id]

        print(f"{rank}. {row['title']} (Search Score: {score})")
        print(f"   | Release Year : {row['release_year']}")
        print(f"   | Director     : {row['director']}")
        print(f"   | Genre        : {row['genre']}")
        print(f"   | Runtime      : {row['runtime']} mins")
        print(f"   | IMDb Rating  : {row['rating']} / 10")
        print(f"   | Metascore    : {row['metascore']} / 100")
        print(f"   | Gross Profit : ${row['gross(M)']} Million")
        print("-" * 50)

    # --- UNIQUE FEATURE: Data-Driven Insights Generation ---
    # We calculate stats based on ALL matched movies for this specific query
    for movie_id, _ in ranked_results:
        row = df.iloc[movie_id]

        # Parse Rating for analytics
        try:
            rating_val = float(row['rating'])
            valid_ratings.append(rating_val)
        except ValueError:
            pass

        # Parse Gross Earnings for analytics
        try:
            gross_val = float(row['gross(M)'])
            valid_earnings.append(gross_val)
            if gross_val > max_earnings:
                max_earnings = gross_val
                highest_earning_movie = row['title']
        except ValueError:
            pass

    # Print the custom insights box
    print("\n SMART SEARCH INSIGHTS FOR THIS QUERY:")
    print("=" * 50)
    print(f"• Total Matched Movies : {len(ranked_results)}")

    if valid_earnings:
        print(f"• Total Box Office Gross : ${sum(valid_earnings):.2f} Million")
        print(f"• Highest Grossing Movie : '{highest_earning_movie}' (${max_earnings:.2f} Million)")
    else:
        print("• Total Box Office Gross : Data Not Available")

    if valid_ratings:
        avg_rating = sum(valid_ratings) / len(valid_ratings)
        print(f"• Average IMDb Rating    : {avg_rating:.2f} / 10")
    print("=" * 50)


# --- Smart Interactive User Input ---
print("\n=== Movie Search Engine Is Ready ===")
user_query = input("Enter your search keywords: ")

tokens = preprocess_text(user_query)
multi_word_flag = False

if len(tokens) > 1:
    multi_word_flag = True
    user_mode = input("Enter search mode (AND / OR): ").strip().upper()
    if user_mode not in ['AND', 'OR']:
        user_mode = 'AND'
else:
    user_mode = 'AND'

# Run the unique search engine structure
ranked_search(user_query, mode=user_mode, is_multi_word=multi_word_flag)

Dataset loaded successfully from GitHub Remote Repository!

=== Movie Search Engine Is Ready ===
Enter your search keywords: Nolan

Results for: 'Nolan'
--------------------------------------------------
1. The Dark Knight (Search Score: 1)
   | Release Year : 2008
   | Director     : Christopher Nolan
   | Genre        : Action
   | Runtime      : 152 mins
   | IMDb Rating  : 9.0 / 10
   | Metascore    : 84 / 100
   | Gross Profit : $534.86 Million
--------------------------------------------------
2. The Dark Knight (Search Score: 1)
   | Release Year : 2008
   | Director     : Christopher Nolan
   | Genre        : Crime
   | Runtime      : 152 mins
   | IMDb Rating  : 9.0 / 10
   | Metascore    : 84 / 100
   | Gross Profit : $534.86 Million
--------------------------------------------------
3. The Dark Knight (Search Score: 1)
   | Release Year : 2008
   | Director     : Christopher Nolan
   | Genre        : Drama
   | Runtime      : 152 mins
   | IMDb Rating  : 9.0 / 10
   | Metasc